<a href="https://colab.research.google.com/github/bquast/colab/blob/master/qwen3_1_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U transformers datasets trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.0 MB/s eta 0:00:00


In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

2.11.0+cu128
True
NVIDIA A100-SXM4-40GB
39.5


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen3-1.7B-Base"

tokenizer = AutoTokenizer.from_pretrained(name)

model = AutoModelForCausalLM.from_pretrained(
    name,
    torch_dtype=torch.float16
).cuda()

print( sum(p.numel() for p in model.parameters()) )

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

1720574976


In [4]:
prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation is a general increase in prices and fall in the purchasing value of money. It's a common economic phenomenon that can be caused by various factors, including increased demand for goods and services, rising production costs, or an increase in the money supply.

User: How does inflation affect the economy?



In [5]:
from datasets import load_dataset, Dataset

stream = load_dataset(
    "HuggingFaceTB/smol-smoltalk",
    split="train",
    streaming=True
)

data = Dataset.from_list(list(stream.take(500)))

def format_example(x):
    text = "\n".join(
        f"{m['role'].title()}: {m['content']}"
        for m in x["messages"]
    )
    return {"text": text + tokenizer.eos_token}

data = data.map(format_example)

print(len(data))
print(data[0]["text"][:1000])

README.md:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

500
User: I need you to edit something for me. This is the text I wrote, 

"Me and my friends have been waiting for a long time to go back to the movies and catch a movie we been waiting on. Last nite I finely went to go buy tickets and when I got to the movie theater the tickets were sold out, so I was pretty pist. We were all pist. So today we deside if we wanted to go to a game instead, but I dont think we will go to a game now."
Assistant: Here's a revised version of your text with some suggested edits to improve grammar, clarity, and overall flow:

"My friends and I have been waiting a long time to return to the movies and see a film we've been eagerly anticipating. Last night, I finally went to buy tickets, but when I arrived at the theater, they were sold out. I was pretty upset, and my friends were too. Today, we discussed the possibility of going to a game instead, but I don't think we'll end up going."

I made the following changes:

- Changed "Me and my friends" to "My frien

In [6]:
data = data.select_columns(["text"])

In [7]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qwen-sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    max_length=512,
    fp16=True,
    logging_steps=5,
    report_to="none"
)

torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    name,
    torch_dtype=torch.float32
).cuda()

print(next(model.parameters()).dtype)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=data,
    processing_class=tokenizer
)

trainer.train()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

torch.float32


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,1.168205
10,1.164644
15,1.109324
20,1.063454
25,1.012749
30,1.113069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32, training_loss=1.111270695924759, metrics={'train_runtime': 145.4315, 'train_samples_per_second': 3.438, 'train_steps_per_second': 0.22, 'total_flos': 2060754807582720.0, 'train_loss': 1.111270695924759, 'entropy': 1.182804775238037, 'num_tokens': 210673.0, 'mean_token_accuracy': 0.6952316135168075, 'epoch': 1.0})

In [8]:
model.eval()

prompt = "User: What causes inflation?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

User: What causes inflation?
Assistant: Inflation is a general increase in the prices of goods and services over time, leading to a decrease in the purchasing power of money. The primary causes of inflation can be categorized into three main types: demand-pull inflation, cost-push inflation, and built-in inflation.

Demand-pull inflation occurs when


In [9]:
model.save_pretrained("qwen3-1.7b-sft")
tokenizer.save_pretrained("qwen3-1.7b-sft")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-1.7b-sft/tokenizer_config.json',
 'qwen3-1.7b-sft/chat_template.jinja',
 'qwen3-1.7b-sft/tokenizer.json')